In [1]:
import pandas as pd
import numpy as np


In [2]:
dataset_raw = pd.read_csv('Data/Product_Normalization_GRI.csv')
dataset_raw.head()


,Room Description,Guest Room Info
0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room
1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room
4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room


In [3]:
def sample_descriptions_by_label(dataset, n_samples_per_class=10, random_state=42):
    """
    Sample Room Descriptions stratified by their Guest Room Info labels
    
    Args:
        dataset: DataFrame with 'Room Description' and 'Guest Room Info'
        n_samples_per_class: Number of descriptions to sample for each label
        random_state: Random seed for reproducibility
    
    Returns:
        List of sampled descriptions and their labels
    """
    # Get unique labels
    labels = dataset['Guest Room Info'].unique()
    
    # Initialize lists to store samples
    sampled_descriptions = []
    sampled_labels = []
    
    print("\nSampling Room Descriptions by Label:")
    print("===================================")
    
    for label in labels:
        # Get all descriptions for this label
        label_data = dataset[dataset['Guest Room Info'] == label]
        
        # Calculate how many samples we can take
        n_available = len(label_data)
        n_to_sample = min(n_samples_per_class, n_available)
        
        if n_to_sample > 0:
            # Sample descriptions for this label
            sampled_data = label_data.sample(
                n=n_to_sample, 
                random_state=random_state
            )
            
            # Add to our lists
            sampled_descriptions.extend(sampled_data['Room Description'].tolist())
            sampled_labels.extend([label] * n_to_sample)
            
            print(f"\n{label}:")
            print(f"- Sampled {n_to_sample} descriptions (out of {n_available} available)")
            # Print first example for each label
            print(f"- Example: {sampled_data['Room Description'].iloc[0]}")
    
    # Create DataFrame of samples
    samples_df = pd.DataFrame({
        'Room Description': sampled_descriptions,
        'Guest Room Info': sampled_labels
    })
    
    # Shuffle the samples
    samples_df = samples_df.sample(
        frac=1, 
        random_state=random_state
    ).reset_index(drop=True)
    
    print(f"\nTotal samples: {len(samples_df)} descriptions")
    print("\nDistribution of labels in sample:")
    print(samples_df['Guest Room Info'].value_counts())
    
    return samples_df



In [4]:
# Sample descriptions
n_samples_per_class = 10
sampled_data = sample_descriptions_by_label(
    dataset_raw, 
    n_samples_per_class=n_samples_per_class
)

# Save samples (optional)
sampled_data
#.to_csv('sampled_descriptions_finetuning_llama_test.csv', index=False)


Sampling Room Descriptions by Label:

Accessible Room:
- Sampled 10 descriptions (out of 1000 available)
- Example: AAA DISCOUNT|ACCESSIBLE 2 QUEENS APPROX 395 SQ FT - JULIET BALCONY ADA ROOM - WHEELCHAIR ACCESSIBLE BATH

Suite:
- Sampled 10 descriptions (out of 1000 available)
- Example: STANDARD RATE 2BR PRES STE|2 BEDROOMS: 2 LIVING ROOMS: FREE BREAKFAST

Executive/Club Suite:
- Sampled 10 descriptions (out of 1000 available)
- Example: ROOM RATE|PREMIER FS EXECUTIVE SUITE KING BED-FLOORS 6-11 CITY AND WATER VW-SEP BEDROOM AND LIVING AREA

Double Bed:
- Sampled 10 descriptions (out of 1000 available)
- Example: FLEXIBLE RATE ROOM ONLY|DOUBLE ROOM - SUITABLE FOR 2 ADULTS

King Bedroom:
- Sampled 10 descriptions (out of 1000 available)
- Example: CCRA PGHP PROMOTIONAL RATE|BROOKLYN BRIDGE KING BED RM 300 SQ FT.

Queen Bedroom:
- Sampled 10 descriptions (out of 1000 available)
- Example: BEST AVAILABLE RATE|QUEEN ROOM W/ SOFA BED AND FRIDGE NON SMOKING FREE WI-FI/HOT BREAKFAST INCLUDE

,Room Description,Guest Room Info
0,2X POINTS|1 KING JR STE MOBILITY/HEARING ACCES...,Junior Suite
1,BEST AVAILABLE RATE|RUN OF HOUSE ROOM ASSIGNED...,Run of the House
2,TL WORLDWIDE PRGM -PREFERRED|1 KING BED CABANA...,Cabana
3,FLEXIBLE RATE|1 KING BED SUPERIOR ROOM 20USD F...,Superior Room
4,GET REAL EXPERIENCE IT ALL|JUNIOR SUITE 2 QUEEN,Junior Suite
...,...,...
345,FLEXIBLE RATE|1 KING BED STUDIO SUITE LAKEVIEW...,Studio Suite
346,DAILY RATE|SUPERIOR COTTAGE-1 KING-VARIOUSVIEW...,Cottage
347,"GOVERNMENT LOCAL|GOVERNMENT LOCAL RATE, LOCAL ...",Classic Suite
348,AAA MEMBERSHIP RATE|1 KING OR 2 QUEEN WHEN YOU...,Run of the House


In [12]:
sampled_data.to_excel('sampled_descriptions_finetuning_llama_test.xlsx')

## Finetuning ##

In [10]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import torch
import pandas as pd

def finetune_llama3(sampled_data, output_dir="finetuned_llama3_1b"):
    """
    Finetune Llama 3.2 1B model on labeled data
    
    Args:
        sampled_data: DataFrame with 'Room Description' and 'Guest Room Info' columns
        output_dir: Directory to save the finetuned model
    """
    # Load model and tokenizer
    model_id = "meta-llama/Llama-3.2-1B-instruct"
    
    print(f"Loading tokenizer from {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    
    print(f"Loading model from {model_id}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,  # Use bfloat16 for better stability
        device_map="auto"  # Automatically distribute across available GPUs
    )
    
    # Prepare dataset
    def format_instruction(row):
        instruction = f"Extract the room type from this hotel description: {row['Room Description']}"
        response = row['Guest Room Info']
        return {
            "text": f"<|user|>\n{instruction}\n<|assistant|>\n{response}</s>"
        }
    
    print("Preparing training data...")
    formatted_data = [format_instruction(row) for _, row in sampled_data.iterrows()]
    train_dataset = Dataset.from_list(formatted_data)
    
    # Configure training
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=2,  # Reduce if OOM errors occur
        gradient_accumulation_steps=8,  # Increase for larger effective batch size
        learning_rate=1e-5,
        weight_decay=0.01,
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="epoch",
        evaluation_strategy="no",
        bf16=True,  # Use bfloat16 precision
        push_to_hub=False,
        gradient_checkpointing=True,  # Enable gradient checkpointing to save memory
        
    )
    
    # Create data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
    
    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )
    
    # Train the model
    print("\nStarting fine-tuning...")
    trainer.train()
    
    # Save the model
    print("\nSaving model...")
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    
    print(f"\nModel saved to {output_dir}")
    return model, tokenizer

In [11]:
# Run fine-tuning
model, tokenizer = finetune_llama3(sampled_data)

Loading tokenizer from meta-llama/Llama-3.2-1B-Instruct...


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading model from meta-llama/Llama-3.2-1B-Instruct...


...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Preparing training data...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_96732/3227290898.py:70: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Starting fine-tuning...


ValueError: No columns in the dataset match the model's forward method signature. The following columns have been ignored: [text]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.